# Create Fields Medal Awards (PRIZE PATTERN)

Fields Medal laureates scraped from the awarding body itself: the
International Mathematical Union (IMU). The Fields Medal is awarded
every four years at the International Congress of Mathematicians;
2-4 medalists per ceremony, awarded since 1936 (1940/1944/1948
skipped due to WWII).

**Awarding body:** International Mathematical Union (IMU) — F4320320877

**Schema notes (prize pattern):**
- One row per (year × laureate). Shared ceremonies produce N rows.
- `amount` and `currency` are **NULL by design**. The Fields Medal carries
  a CA$15k stipend that is nominal in OpenAlex terms — we treat it as a
  non-monetary prestige prize. Step 6.7's amount-coverage check is waived
  here with an explicit note (see verification cells).
- `lead_investigator` = the laureate themselves (given_name + family_name
  populated). `affiliation` is **NULL** because IMU does not publish the
  affiliation-when-awarded in any structured form on their site; we do not
  backfill it from Wikipedia because that would split provenance across
  two sources. `co_lead_investigator` and `investigators` are NULL.
- `description` = the IMU citation paragraph when available. IMU only
  publishes per-laureate citations for the modern cohorts (2014, 2018,
  2022 — 12 of 64 rows). Older rows have NULL description. Perelman (2006)
  declined the medal; his row's `description` is prefixed
  `'Declined the medal. '` so consumers can distinguish.
- `funder_award_id` = `fields-{year}-{lowercased-surname-slug}` —
  duplicate-slug detection in the upload script raises before parquet write.

**Prerequisites:**
- Run `scripts/local/fields_medal_to_s3.py` first to scrape the IMU site
  and upload the parquet to S3.

**Data source:** https://www.mathunion.org/imu-awards/fields-medal
**S3 location:** `s3a://openalex-ingest/awards/fields_medal/fields_medal_laureates.parquet`


## Step 1: Create staging table from S3

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.fields_medal_raw
USING delta
AS
SELECT *, current_timestamp() AS databricks_ingested_at
FROM parquet.`s3a://openalex-ingest/awards/fields_medal/fields_medal_laureates.parquet`;

In [ ]:
%sql
SELECT COUNT(*) FROM openalex.awards.fields_medal_raw;

## Step 1.5: Inspect raw + amount/currency note

**This is a non-monetary prize.** `amount` and `currency` will be NULL on
the output side by design — the Fields Medal CA$15k stipend is nominal
and we treat the medal as a prestige prize per the prize pattern in
how-to-add-a-funder.md. No amount-field discovery scan is needed; the
columns are author-controlled (we wrote the scraper in `scripts/local/`).

Sanity-check the staging table anyway:

In [ ]:
%sql
DESCRIBE openalex.awards.fields_medal_raw;

In [ ]:
%sql
SELECT * FROM openalex.awards.fields_medal_raw LIMIT 5;

In [ ]:
%sql
-- Coverage check on key fields.
-- NOTE: affiliation_when_awarded expected = 0 (IMU does not publish it).
-- citation expected ~12/64 (only 2014/2018/2022 have per-laureate
-- citation paragraphs on the IMU site).
SELECT
    COUNT(*) AS total_rows,
    COUNT(year) AS has_year,
    COUNT(medalist_name) AS has_name,
    COUNT(affiliation_when_awarded) AS has_aff,
    COUNT(personal_url) AS has_personal_url,
    COUNT(citation) AS has_citation,
    SUM(CASE WHEN declined THEN 1 ELSE 0 END) AS declined_count,
    MIN(year) AS min_year,
    MAX(year) AS max_year,
    COUNT(DISTINCT year) AS distinct_years
FROM openalex.awards.fields_medal_raw;

## Step 1.6: Fail-fast — verify the IMU funder row exists

The transform in Step 2 does `CROSS JOIN openalex.common.funder WHERE
funder_id = 4320320877`. If that row isn't present, the CROSS JOIN
silently emits zero rows and the insert in Step 3 looks like it
succeeded. Assert presence here before transforming.

In [ ]:
%sql
-- Must return exactly 1 row with display_name = 'International Mathematical Union'.
-- If 0 rows, STOP and flag — the funder is missing from openalex.common.funder.
SELECT funder_id, display_name, ror_id, doi
FROM openalex.common.funder
WHERE funder_id = 4320320877;

## Step 2: Transform to award schema

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.fields_medal_awards
USING delta
AS
WITH funder_resolved AS (
    SELECT funder_id, display_name, ror_id, doi
    FROM openalex.common.funder
    WHERE funder_id = 4320320877  -- International Mathematical Union
)
SELECT
    -- Unique ID per (funder, slug). Slug is year-surname (e.g. '2014-mirzakhani').
    abs(xxhash64(CONCAT(
        CAST(f.funder_id AS STRING), ':fields:', n.slug
    ))) % 9000000000 AS id,
    -- "Fields Medal {year} — {name}"
    CONCAT('Fields Medal ', TRY_CAST(n.year AS STRING), ' — ', n.medalist_name) AS display_name,
    CASE
        WHEN n.declined AND n.citation IS NOT NULL THEN CONCAT('Declined the medal. ', n.citation)
        WHEN n.declined THEN 'Declined the medal.'
        ELSE n.citation
    END AS description,
    f.funder_id,
    CONCAT('fields-', n.slug) AS funder_award_id,
    CAST(NULL AS DOUBLE) AS amount,           -- non-monetary prize, waived per §6.7
    CAST(NULL AS STRING) AS currency,         -- no currency for non-monetary prize
    struct(
        CONCAT('https://openalex.org/F', CAST(f.funder_id AS STRING)) AS id,
        f.display_name,
        f.ror_id,
        f.doi
    ) AS funder,
    'prize' AS funding_type,
    'Fields Medal' AS funder_scheme,
    'imu_fields_medal' AS provenance,
    -- ICM ceremonies are typically held in August; use Aug 1 of the award year as the canonical date.
    TRY_TO_DATE(CONCAT(TRY_CAST(n.year AS STRING), '-08-01'), 'yyyy-MM-dd') AS start_date,
    TRY_TO_DATE(CONCAT(TRY_CAST(n.year AS STRING), '-08-01'), 'yyyy-MM-dd') AS end_date,
    TRY_CAST(n.year AS INT) AS start_year,
    TRY_CAST(n.year AS INT) AS end_year,
    -- The laureate IS the lead investigator. Affiliation/country left NULL
    -- because IMU does not publish affiliation-when-awarded in structured form.
    struct(
        n.given_name AS given_name,
        n.family_name AS family_name,
        CAST(NULL AS STRING) AS orcid,
        CAST(NULL AS DATE) AS role_start,
        struct(
            n.affiliation_when_awarded AS name,
            CAST(NULL AS STRING) AS country,
            CAST(NULL AS ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>) AS ids
        ) AS affiliation
    ) AS lead_investigator,
    CAST(NULL AS STRUCT<
        given_name:STRING, family_name:STRING, orcid:STRING, role_start:DATE,
        affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>
    >) AS co_lead_investigator,
    CAST(NULL AS ARRAY<STRUCT<
        given_name:STRING, family_name:STRING, orcid:STRING, role_start:DATE,
        affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>
    >>) AS investigators,
    n.source_url AS landing_page_url,
    CAST(NULL AS STRING) AS doi,
    CONCAT('https://api.openalex.org/works?filter=awards.id:G',
           CAST(abs(xxhash64(CONCAT(
               CAST(f.funder_id AS STRING), ':fields:', n.slug
           ))) % 9000000000 AS STRING)) AS works_api_url,
    current_timestamp() AS created_date,
    current_timestamp() AS updated_date
FROM openalex.awards.fields_medal_raw n
CROSS JOIN funder_resolved f
WHERE n.slug IS NOT NULL AND n.year IS NOT NULL;

## Step 3: Insert into openalex_awards_raw at priority 50

In [ ]:
%sql
DELETE FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'imu_fields_medal' AND priority = 50;

INSERT INTO openalex.awards.openalex_awards_raw
SELECT
    id, display_name, description, funder_id, funder_award_id,
    amount, currency, funder, funding_type, funder_scheme, provenance,
    start_date, end_date, start_year, end_year,
    lead_investigator, co_lead_investigator, investigators,
    landing_page_url, doi, works_api_url,
    created_date, updated_date,
    50 AS priority
FROM openalex.awards.fields_medal_awards;

## Step 6: Verification

The runbook's §6.7 fail-fast check is **waived** here for amount/currency
coverage because the Fields Medal is a non-monetary prize. We still
verify row count, year coverage, funder consistency, and that the
amount column is uniformly NULL (i.e. no accidental partial-fill that
would corrupt aggregates downstream).

In [ ]:
%sql
SELECT COUNT(*) AS total_fields_medal_award_rows FROM openalex.awards.fields_medal_awards;

In [ ]:
%sql
-- §6.2 Schema validation on the transformed table
DESCRIBE openalex.awards.fields_medal_awards;

In [ ]:
%sql
-- §6.3 Data completeness (runbook canonical query)
-- NOTE: pct_amount expected to be 0% — Fields Medal is a non-monetary
-- prize and amount is NULL by design (§6.7 waiver).
-- NOTE: pct_description expected ~19% (12/64) — IMU only publishes
-- per-laureate citations for 2014/2018/2022 cohorts.
SELECT
    COUNT(*) AS total,
    COUNT(display_name) AS has_title,
    COUNT(description) AS has_description,
    COUNT(amount) AS has_amount,
    COUNT(start_date) AS has_start_date,
    COUNT(lead_investigator) AS has_pi,
    ROUND(try_divide(COUNT(display_name), COUNT(*)) * 100.0, 1) AS pct_title,
    ROUND(try_divide(COUNT(description), COUNT(*)) * 100.0, 1) AS pct_description,
    ROUND(try_divide(COUNT(start_date), COUNT(*)) * 100.0, 1) AS pct_dates
FROM openalex.awards.fields_medal_awards;

In [ ]:
%sql
-- §6.7 waiver: amount/currency should be 100% NULL by design.
SELECT
    COUNT(*) AS total,
    COUNT(amount) AS has_amount,
    COUNT(currency) AS has_currency,
    COUNT(DISTINCT funder.display_name) AS distinct_funders,
    collect_set(funder.display_name) AS funders,
    collect_set(funding_type) AS funding_types
FROM openalex.awards.fields_medal_awards;

In [ ]:
%sql
-- Sample inspection
SELECT id, SUBSTRING(display_name, 1, 60) AS title,
       start_year, funder_award_id,
       lead_investigator.given_name AS given,
       lead_investigator.family_name AS family,
       SUBSTRING(lead_investigator.affiliation.name, 1, 50) AS affiliation,
       SUBSTRING(description, 1, 80) AS citation_preview
FROM openalex.awards.fields_medal_awards
ORDER BY start_year DESC, family
LIMIT 12;

In [ ]:
%sql
-- Year distribution (quadrennial since 1950; 1942/1946 skipped due to WWII)
SELECT start_year, COUNT(*) AS medalists
FROM openalex.awards.fields_medal_awards
GROUP BY start_year
ORDER BY start_year;

In [ ]:
%sql
-- Funder struct populated correctly?
SELECT funder.id, funder.display_name, funder.ror_id, funder.doi, COUNT(*) AS rows
FROM openalex.awards.fields_medal_awards
GROUP BY funder.id, funder.display_name, funder.ror_id, funder.doi;